# March Mania · Margin robustness
**Milestone 11 — preserve the completed recovery → engineer two representations → fixed comparisons → report.**

Round10 completed: seven ratings, twelve comparisons and a saved scientific report. Neither declared comparison passed its expansion rule. Do not rerun recovery.

This notebook tests residual-robust and compressed-margin ratings, not different tournament algorithms. The 16-input reference stays unchanged. At most **14 new rating fits + 12 new classifiers**. Four references and seven base snapshots are reused. All 2016–2019 seasons are already-used exploratory history. Read `RESEARCH_PLAN.md` for sources, formulas and limitations.

Use **Python (March Mania)**. No installations, cloud API calls, Git changes or submissions. Keep all prior folders.

In [ ]:
from pathlib import Path
import sys,json
import pandas as pd
import plotly.io as pio
from IPython.display import display,FileLink
KIT=Path.cwd().resolve()
if not (KIT/'run_round11.py').is_file():
    KIT=Path.home()/'march_margin_research'
assert (KIT/'run_round11.py').is_file(), 'Open this notebook from march_margin_research.'
sys.path.insert(0,str(KIT))
from run_round11 import run_stage
from margin_plots import figures
pio.renderers.default='plotly_mimetype'
pd.set_option('display.precision',7)
print('Kernel:',sys.executable)
print('Kit:',KIT)
print('No recovery or old experiment will be rerun.')

## 1. Completed work is evidence, not a reason to refit
The uploaded recovery archive contains all twelve Brier values in its evaluation log and a receipt for the full scientific ZIP. The first scientific stage verifies that existing ZIP by SHA-256 and checks all logged scores against it before fitting anything. That full archive will be included in this milestone’s return package.

In [ ]:
previous=pd.read_csv(KIT/'evidence/round10/log_metrics.csv',float_precision='round_trip')
wide=previous.pivot(index='Season',columns='recipe',values='brier')
wide['ability_delta']=wide.anchor_bt-wide.anchor
wide['uncertainty_delta']=wide.anchor_bt_uncertainty-wide.anchor_bt
display(wide.round(7))
print('Mean ability delta:',wide.ability_delta.mean())
print('Mean uncertainty delta:',wide.uncertainty_delta.mean())
print('These values were parsed from your recovery log, not fitted here.')

## 2. Build two new representations
**Residual-robust rating:** minimize
\[\sum_g H_{15}\big(m_g-(s_{w_g}-s_{l_g}+h\,v_g)\big)+10\|\theta\|_2^2.\]
The Huber loss uses squared residuals near zero and linear growth outside ±15 points. A predicted 30-point win is not downweighted simply because it is a blowout.

**Compressed-margin rating:** regress \(15\tanh(m_g/15)\) on the same opponent/home design with ridge alpha 20. This deliberately compresses the observed margin; it is a different hypothesis and can lose useful information.

The 15-point constants are fixed research choices, not claimed optimal. Games through day 132 only. Build potential seeded pairs before attaching tournament labels. **Preparation cap: 300 seconds.** Each accepted rating has a separate checkpoint. No unbounded solver retry.

In [ ]:
run_stage('prepare',max_seconds=300)
RUN=Path(json.loads((KIT/'reports/latest_run.json').read_text())['run_dir'])
print(json.dumps(json.loads((RUN/'prepare.json').read_text()),indent=2))
display(pd.read_csv(RUN/'rating_diagnostics.csv'))
display(pd.read_csv(RUN/'feature_registry.csv').query('new_candidate == True'))

## 3. Check the unchanged reference and support
The prior best submission uses a stronger, different production recipe. These compact-reference comparisons only test the representation under a fixed diagnostic recipe. A failure here is not proof of universal feature uselessness. The new ratings are estimated from regular-season data, not tournament targets.

In [ ]:
display(pd.read_csv(RUN/'prior_replay.csv'))
display(pd.read_csv(RUN/'coverage.csv'))
print(json.dumps(json.loads((RUN/'prior_results_check.json').read_text()),indent=2))

## 4. Four configurations × four seasons
`anchor` (16), `anchor_huber` (17), `anchor_compressed` (17), `anchor_both` (18).

Men only. Train on 2013 through the prior season; evaluate main-draw 2016–2019. C=0.1, no intercept, mirrored orientations, physical-game weighting and train-only RMS scaling remain fixed. Only training-constant columns may be dropped. **Twelve new classifiers, four references replayed; evaluation cap 180 seconds.**

In [ ]:
run_stage('evaluate',max_seconds=180)
metrics=pd.read_csv(RUN/'metrics.csv')
display(metrics[['Season','recipe','brier','log_loss','delta_vs_anchor','source']].round(7))
display(pd.read_csv(RUN/'ablations.csv').round(7))
print(json.dumps(json.loads((RUN/'decisions.json').read_text()),indent=2))

## 5. Export before inline plotting
The primary is `huber_given_anchor`. `compression_given_huber` is secondary; it cannot replace a failed primary. Consider further work only at mean Brier delta ≤ −0.0005, improvement in ≥3/4 seasons and worst deterioration ≤ +0.003. This is a compute-allocation rule, not significance. Nothing automatically runs next.

The report is saved before the inline charts so a display problem does not lose results. **Report cap: 120 seconds.**

In [ ]:
run_stage('report',max_seconds=120)
record=json.loads((KIT/'reports/latest_report.json').read_text())
print(json.dumps(record,indent=2))
display(FileLink(str(Path(record['return_zip']).relative_to(KIT))))
display(FileLink(str(Path(record['html']).relative_to(KIT))))

## 6. Interactive evidence
Ten Plotly figures: previous completed results, two mathematical mechanisms, convergence, team profiles, Brier, controlled effects, calibration and training-only overlap. Use hover for exact values. No chart implies causality or fresh-holdout validity.

In [ ]:
plots=figures(RUN,KIT/'evidence/round10')
assert len(plots)==10
for fig in plots[:5]:
    fig.show()

In [ ]:
for fig in plots[5:]:
    fig.show()
print('Save with Ctrl+S. Return reports/milestone_11_return.zip. Stop here.')